In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from torch.optim import AdamW
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [ ]:
## LOAD DATA

# define Data class for torch DataLoader
class Data(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        item = self.examples.iloc[idx]
        label = "yes" if item["label"] == 1 else "no"
        input = item['input']
        return input, label

train_data_df = pd.read_csv('/path/to/T_standard.csv') # load training data
train_data = DataLoader(Data(train_data_df), batch_size=64, shuffle=True) # apply DataLoader for random batching

In [ ]:
## LOAD MODEL

model_name = "google/flan-t5-large"
tokeniser = T5Tokenizer.from_pretrained(model_name) # load 'untrained' FLAN-T5-Large
model = T5ForConditionalGeneration.from_pretrained(model_name,
                                                   torch_dtype=torch.bfloat16,
                                                   device_map='cuda')
optimiser = AdamW(model.parameters(), lr=5e-4)
model.gradient_checkpointing_enable()

In [ ]:
model.train()
for epoch in range(20):
    epoch_loss = 0
    for batch in tqdm(train_data):
        texts, labels = batch

        inputs = tokeniser(texts, return_tensors="pt", truncation=True, padding=True, max_length=512).to('cuda') # tokenise texts
        labels = tokeniser(labels, return_tensors="pt", truncation=True, padding=True, max_length=2).input_ids.to("cuda") # tokenise labels

        outputs = model(**inputs, labels=labels) # obtain model outputs
        loss = outputs.loss # obtain loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        optimiser.zero_grad()
        epoch_loss += loss.item()

    print(f"Epoch {epoch} Loss: {epoch_loss / len(train_data_df):.4f}")

model.save_pretrained('ts1')